# Demonstrating Qiskit Aer Features with a 2x2 Sudoku Puzzle

This notebook demonstrates the various features of Qiskit Aer using a 2x2 Sudoku puzzle with all 4 cells missing. The puzzle is solved using the `ExactCoverQuantumSolver` with pattern encoding.

In [1]:
# Import necessary libraries
from sudoku_nisq import QSudoku
from sudoku_nisq.solvers import ExactCoverQuantumSolver
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the 2x2 Sudoku puzzle with all cells missing
puzzle = QSudoku.generate(size=2, num_missing_cells=4)
puzzle.set_solver(ExactCoverQuantumSolver, encoding="pattern")

In [ ]:
# Run the puzzle on Aer with default settings
result = puzzle.run_aer(shots=1024)
print("Default Aer Simulation:")
print(f"Measured {len(result.get_counts())} unique outcomes")

In [ ]:
# Specify simulation method: statevector
result = puzzle.run_aer(
    shots=2048,
    method="statevector",
    optimization_level=2
)
print("Statevector Simulation:")
print(f"Measured {len(result.get_counts())} unique outcomes")

In [ ]:
# Add noise model and run density matrix simulation
noise = NoiseModel()
noise.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 1), ['u1', 'u2', 'u3', 'h', 's', 't']
)
noise.add_all_qubit_quantum_error(
    depolarizing_error(0.02, 2), ['cx', 'cz', 'swap']
)

result = puzzle.run_aer(
    shots=4096,
    method="density_matrix",
    noise_model=noise
)
print("Density Matrix Simulation with Noise:")
print(f"Measured {len(result.get_counts())} unique outcomes")

In [ ]:
# GPU acceleration example (if available)
from sudoku_nisq.providers import AerProvider

provider = AerProvider()
info = provider.query_available_devices()
if info['has_gpu']:
    result = puzzle.run_aer(
        shots=2048,
        method="statevector",
        device="GPU",
        precision="single"
    )
    print("GPU-Accelerated Simulation:")
    print(f"Measured {len(result.get_counts())} unique outcomes")
else:
    print("GPU not available. Skipping GPU-accelerated simulation.")